<a href="https://colab.research.google.com/github/lizzietm/Parapet-Workbench/blob/main/Exa_Python_Researcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exa Researcher Python Colab Notebook

## Setup

First, let's install the necessary libraries. Run this cell to install `exa_py` and update `openai` to the latest version.

In [1]:
!pip install exa_py openai --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.7 MB/s eta 0:00:00


Now, let's import the required libraries and set up our API keys. Make sure to enter your API keys in the appropriate places.

In [ ]:
import os
import exa_py
from openai import OpenAI

# Set your API keys here
os.environ['EXA_API_KEY'] = '...'
os.environ['OPENAI_API_KEY'] = '...'

EXA_API_KEY = os.environ.get('EXA_API_KEY')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

exa = exa_py.Exa(EXA_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# Test topics
SAMA_TOPIC = 'Sam Altman'
ART_TOPIC = 'renaissance art'

## Helper Functions

Let's define our helper functions for interacting with the OpenAI API and generating search queries.

In [ ]:
def get_llm_response(system='You are a helpful assistant.', user='', temperature=1, model='gpt-3.5-turbo'):
    completion = openai_client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ]
    )
    return completion.choices[0].message.content

def generate_search_queries(topic, n):
    user_prompt = f"""I'm writing a research report on {topic} and need help coming up with diverse search queries.
Please generate a list of {n} search queries that would be useful for writing a research report on {topic}. These queries can be in various formats, from simple keywords to more complex phrases. Do not add any formatting or numbering to the queries."""

    completion = get_llm_response(
        system='The user will ask you to help generate some search queries. Respond with only the suggested queries in plain text with no extra formatting, each on its own line.',
        user=user_prompt,
        temperature=1
    )
    return [s.strip() for s in completion.split('\n') if s.strip()][:n]

## Exa Search Function

Now, let's define the function that performs searches using Exa's API.

In [ ]:
def get_search_results(queries, links_per_query=2):
    results = []
    for query in queries:
        search_response = exa.search_and_contents(query,
            num_results=links_per_query,
            use_autoprompt=False
        )
        results.extend(search_response.results)
    return results

## Report Synthesis Function

This function will use the OpenAI API to synthesize the search results into a coherent report.

In [ ]:
def synthesize_report(topic, search_contents, content_slice=750):
    input_data = '\n'.join([f"--START ITEM--\nURL: {item.url}\nCONTENT: {item.text[:content_slice]}\n--END ITEM--\n" for item in search_contents])
    return get_llm_response(
        system='You are a helpful research assistant. Write a report according to the user\'s instructions.',
        user=f'Input Data:\n{input_data}Write a two paragraph research report about {topic} based on the provided information. Include as many sources as possible. Provide citations in the text using footnote notation ([#]). First provide the report, followed by a single "References" section that lists all the URLs used, in the format [#] <url>.',
        # model='gpt-4'  # want a better report? use gpt-4 (but it costs more)
    )

## Main Researcher Function

Now, let's combine all of our functions into the main researcher function.

In [ ]:
def researcher(topic):
    print(f'Starting research on topic: "{topic}"')

    search_queries = generate_search_queries(topic, 3)
    print("Generated search queries:", search_queries)

    search_results = get_search_results(search_queries)
    print(f"Found {len(search_results)} search results. Here's the first one:", search_results[0])

    print("Synthesizing report...")
    report = synthesize_report(topic, search_results)

    return report

## Run Examples

Finally, let's run our researcher on multiple topics to see it in action.

In [ ]:
def run_examples():
    topics = [SAMA_TOPIC, ART_TOPIC, "artificial intelligence ethics", "climate change solutions"]

    for topic in topics:
        print(f"\n{'='*50}\nResearching: {topic}\n{'='*50}\n")
        report = researcher(topic)
        print(report)
        print("\n")

# Run the examples
run_examples()


Researching: Sam Altman

Starting research on topic: "Sam Altman"
Generated search queries: ['Sam Altman background bio', 'Sam Altman Y Combinator contributions', 'Sam Altman latest projects and investments']
Found 6 search results. Here's the first one: Title: Sam Altman - Wikipedia
URL: https://en.wikipedia.org/wiki/Sam_Altman
ID: https://en.wikipedia.org/wiki/Sam_Altman
Score: 0.2533408999443054
Published Date: 2009-04-19
Author: None
Text: From Wikipedia, the free encyclopedia

Sam Altman Altman in 2019 Born Samuel H. Altman April 22, 1985 (age 37) Chicago, Illinois, U.S. Education Stanford University (dropped out) Occupation Entrepreneur Known for Loopt, Y Combinator, OpenAI Title CEO of OpenAI LP and former president of Y Combinator Website Official website 
 Samuel H. Altman ( AWLT-mən ; born April 22, 1985) is an American entrepreneur, investor, programmer, and blogger. [1] He is the CEO of OpenAI and the former president of Y Combinator. [2] [3]

Early life and education[edit

This cell will run the researcher on four different topics: Sam Altman, Renaissance Art, AI Ethics, and Climate Change Solutions. It will print out the generated queries, search results, and final synthesized report for each topic.

Feel free to modify the `topics` list in the `run_examples()` function to research any other topics you're interested in!